In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, IntegerType, FloatType, DoubleType

In [70]:
# To allow automatic schemaInference while reading


spark = SparkSession \
    .builder \
    .appName("Streaming Process Files") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .master("local[*]") \
    .getOrCreate()


spark.conf.set("spark.sql.streaming.schemaInference", True)

# Create the streaming_df to read from input directory
streaming_df = spark.read\
    .format("json") \
    .option("maxFilesPerTrigger", 1) \
    .load("../parte3_streaming/streaming_sample.json")

In [72]:
streaming_df.printSchema()
streaming_df.show(truncate=False)

root
 |-- DayOfWeek: string (nullable = true)
 |-- EnergyConsumption: double (nullable = true)
 |-- HVACUsage: string (nullable = true)
 |-- Holiday: string (nullable = true)
 |-- Hour: long (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- LightingUsage: string (nullable = true)
 |-- Month: long (nullable = true)
 |-- Occupancy: long (nullable = true)
 |-- RenewableEnergy: double (nullable = true)
 |-- SquareFootage: double (nullable = true)
 |-- Temperature: double (nullable = true)

+---------+-----------------+---------+-------+----+-------------+-------------+-----+---------+---------------+---------------+-------------+
|DayOfWeek|EnergyConsumption|HVACUsage|Holiday|Hour|Humidity     |LightingUsage|Month|Occupancy|RenewableEnergy|SquareFootage  |Temperature  |
+---------+-----------------+---------+-------+----+-------------+-------------+-----+---------+---------------+---------------+-------------+
|Thursday |84.7785708985    |Off      |Yes    |5   |30.015975    |O

In [79]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, sum
# Consumo energético promedio por mes
df_avg_consumption = streaming_df.groupBy("Month").agg(
    avg("EnergyConsumption").alias("AvgConsumption"),
    sum("EnergyConsumption").alias("TotalConsumption")
).orderBy (col("Month").asc())

df_avg_consumption.show()

+-----+-----------------+------------------+
|Month|   AvgConsumption|  TotalConsumption|
+-----+-----------------+------------------+
|    1|76.14319552658561|12715.913652939797|
|    2|77.15083606253063|3780.3909670640005|
|    3|77.67882692870845| 6447.342635082801|
|    4|77.95528781594474| 5924.601874011801|
|    5|75.68070809735072|5221.9688587171995|
|    6|74.31175912667537| 5424.758416247302|
|    7|76.84387357926423| 6224.353759920403|
|    8|78.75078730083999| 6693.816920571399|
|    9|75.57836507440852| 6197.425936101499|
|   10|76.53269148032436|   5969.5499354653|
|   11|76.06207373223442| 4867.972718863003|
|   12|77.34407260450644| 7192.998752219099|
+-----+-----------------+------------------+



In [83]:
from pyspark.sql.functions import max, min
# Máximo y mínimo de temperatura por día de la semana
df_temp_stats = streaming_df.groupBy("DayOfWeek").agg(
    max("Temperature").alias("MaxTemp"),
    min("Temperature").alias("MinTemp")
).orderBy (col("DayOfWeek").asc())
df_temp_stats.show()



+---------+---------+---------+
|DayOfWeek|  MaxTemp|  MinTemp|
+---------+---------+---------+
|   Friday|29.998671|20.007565|
|   Monday|29.998671|20.007565|
| Saturday|29.998671|20.007565|
|   Sunday|29.998671|20.007565|
| Thursday|29.998671|20.007565|
|  Tuesday|29.998671|20.007565|
|Wednesday|29.998671|20.007565|
+---------+---------+---------+



In [85]:
# Filtrar horas con alto consumo energético (> 85)
df_high_consumption = streaming_df.filter(col("EnergyConsumption") > 85)

df_high_consumption.show()


+---------+-----------------+---------+-------+----+-------------+-------------+-----+---------+---------------+---------------+-------------+
|DayOfWeek|EnergyConsumption|HVACUsage|Holiday|Hour|     Humidity|LightingUsage|Month|Occupancy|RenewableEnergy|  SquareFootage|  Temperature|
+---------+-----------------+---------+-------+----+-------------+-------------+-----+---------+---------------+---------------+-------------+
|   Friday|    85.2313905536|      Off|     No|  10| 48.097898971|           On|    7|        5|  23.0189621381| 1506.721568008|24.5454047028|
| Thursday|     94.525334344|      Off|     No|   9|    30.015975|           On|    3|        7|  13.9062734722|1867.3542370192|28.8492621233|
|Wednesday|    85.3884173559|      Off|    Yes|  14| 54.963983642|           On|    6|        0|   9.0668732202|    1000.512661|    29.998671|
|   Friday|    86.2759410095|      Off|    Yes|  21|41.6449654957|           On|    1|        7|   8.4538954657|1185.8941762127|26.6601213738|

In [87]:
# Filtrar días festivos con HVAC encendido
df_holiday_hvac = streaming_df.filter(
    (col("Holiday") == "Yes") & 
    (col("HVACUsage") == "On")
)
df_holiday_hvac.show()


+---------+-----------------+---------+-------+----+-------------+-------------+-----+---------+---------------+---------------+-------------+
|DayOfWeek|EnergyConsumption|HVACUsage|Holiday|Hour|     Humidity|LightingUsage|Month|Occupancy|RenewableEnergy|  SquareFootage|  Temperature|
+---------+-----------------+---------+-------+----+-------------+-------------+-----+---------+---------------+---------------+-------------+
|  Tuesday|    83.5017953085|       On|    Yes|  13|44.9933197133|           On|    1|        9|  13.8094757588|1733.4801540112|25.2951587507|
| Thursday|    68.4910363371|       On|    Yes|   2|    30.015975|          Off|    6|        4|   6.1012666915|1062.6907435836|21.0084661267|
| Thursday|    74.4499844916|       On|    Yes|   2|50.6510266507|          Off|    2|        2|   6.1069436066|1290.9855914872|23.6292620536|
|   Monday|    83.9106416346|       On|    Yes|  12|48.6707139858|          Off|    8|        2|   5.3535295769|1227.8621795121|    29.998671|

In [88]:
# Clasificar consumo energético
df_with_category = streaming_df.withColumn(
    "ConsumptionCategory",
    when(col("EnergyConsumption") < 80, "Bajo")
    .when(col("EnergyConsumption") < 90, "Medio")
    .otherwise("Alto")
)
df_with_category.show()

+---------+-----------------+---------+-------+----+-------------+-------------+-----+---------+---------------+---------------+-------------+-------------------+
|DayOfWeek|EnergyConsumption|HVACUsage|Holiday|Hour|     Humidity|LightingUsage|Month|Occupancy|RenewableEnergy|  SquareFootage|  Temperature|ConsumptionCategory|
+---------+-----------------+---------+-------+----+-------------+-------------+-----+---------+---------------+---------------+-------------+-------------------+
| Thursday|    84.7785708985|      Off|    Yes|   5|    30.015975|          Off|    4|        7|  10.3205021785|1424.6787006567|23.1150588513|              Medio|
| Thursday|     62.777773282|      Off|    Yes|   5|44.2251048101|           On|    6|        2|  18.3757083613|1805.7461672992|28.4162414944|               Bajo|
|   Friday|    85.2313905536|      Off|     No|  10| 48.097898971|           On|    7|        5|  23.0189621381| 1506.721568008|24.5454047028|              Medio|
| Thursday|     94.525

In [89]:
# Calcular eficiencia energética
df_with_efficiency = streaming_df.withColumn(
    "EnergyEfficiency",
    col("RenewableEnergy") / col("EnergyConsumption") * 100
)
df_with_efficiency.show()

+---------+-----------------+---------+-------+----+-------------+-------------+-----+---------+---------------+---------------+-------------+--------------------+
|DayOfWeek|EnergyConsumption|HVACUsage|Holiday|Hour|     Humidity|LightingUsage|Month|Occupancy|RenewableEnergy|  SquareFootage|  Temperature|    EnergyEfficiency|
+---------+-----------------+---------+-------+----+-------------+-------------+-----+---------+---------------+---------------+-------------+--------------------+
| Thursday|    84.7785708985|      Off|    Yes|   5|    30.015975|          Off|    4|        7|  10.3205021785|1424.6787006567|23.1150588513|  12.173479771033273|
| Thursday|     62.777773282|      Off|    Yes|   5|44.2251048101|           On|    6|        2|  18.3757083613|1805.7461672992|28.4162414944|  29.271041963778583|
|   Friday|    85.2313905536|      Off|     No|  10| 48.097898971|           On|    7|        5|  23.0189621381| 1506.721568008|24.5454047028|  27.007610680273864|
| Thursday|     

In [91]:

from pyspark.sql.functions import count
# Consumo por uso de HVAC y Lighting
df_usage_stats = streaming_df.groupBy("HVACUsage", "LightingUsage").agg(
    avg("EnergyConsumption").alias("AvgConsumption"),
    count("*").alias("RecordsCount")
)
df_usage_stats.show()

+---------+-------------+-----------------+------------+
|HVACUsage|LightingUsage|   AvgConsumption|RecordsCount|
+---------+-------------+-----------------+------------+
|      Off|          Off|76.38937216518391|         248|
|       On|           On|76.72354683085248|         242|
|       On|          Off|77.41419067138315|         262|
|      Off|           On|76.07626548898914|         248|
+---------+-------------+-----------------+------------+



In [114]:
# Porcentaje de tiempo con HVAC encendido por mes
df_hvac_usage = streaming_df.groupBy("Month").agg(
    (sum(when(col("HVACUsage") == "On", 1).otherwise(0)) / count("*") * 100).alias("HVACOnPercentage")
).orderBy (col("Month").asc())

df_hvac_usage.show()
    






+-----+------------------+
|Month|  HVACOnPercentage|
+-----+------------------+
|    1| 47.90419161676647|
|    2| 48.97959183673469|
|    3|50.602409638554214|
|    4| 57.89473684210527|
|    5|44.927536231884055|
|    6|52.054794520547944|
|    7|  41.9753086419753|
|    8| 48.23529411764706|
|    9| 56.09756097560976|
|   10| 52.56410256410257|
|   11|           51.5625|
|   12| 53.76344086021505|
+-----+------------------+

